# Séance 4 : Outils Python pour l'analyse EEG

PSY2007D — Laboratoire 1, automne 2026

Les briques Python qu'on retrouve dans tous les notebooks d'analyse du cours.

| Partie | Contenu |
|---|---|
| 0 | Vérifier l'environnement |
| 1 | Chemins, listes, dictionnaires |
| 2 | L'EEG est un tableau NumPy |
| 3 | Fonctions, boucles, tableau de résultats |
| 4 | Aligner l'EEG et la plateforme de force |

> **Données.** Les données du projet 2026 ne sont pas encore enregistrées. Ce notebook utilise un jeu public (*kiloword*, parties 2-3), des données de marche simulées (partie 4) et des sujets fictifs (partie 1).

**Blocs** — Type 1 : exécuter et lire · Type 2 : compléter les lignes marquées `<---` · Type 3 : optionnel.

<details><summary>Colab ou VS Code ?</summary>

- **Colab** : `Fichier → Enregistrer une copie dans Drive` avant de commencer.
- **VS Code** : ouvrir le notebook et choisir le noyau `env_meeg` (en haut à droite).
- Les cellules grises repliées (dans Colab) contiennent du code utilitaire : il suffit de les exécuter.
</details>

<details><summary>Pourquoi ces briques ?</summary>

Les notebooks d'analyse (ceux de 2025, dans la branche `2025` du dépôt, et ceux qui seront écrits pour 2026) suivent tous la même logique : on teste chaque étape **sur un sujet**, une **fonction** regroupe les étapes, on l'applique à **tous les sujets**, et les résultats vont dans un **tableau CSV** utilisé pour les statistiques et l'apprentissage machine.
</details>

## 0. Vérifier l'environnement

### Bloc Type 1

Une erreur `ModuleNotFoundError` signifie qu'un paquet manque dans l'environnement actif.

In [ ]:
# ---- Installation de MNE (Colab seulement ; ne fait rien en local)
import sys
if "google.colab" in sys.modules:
    !pip install -q mne mne-bids

In [ ]:
# ---- Imports
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

mne.set_log_level("WARNING")
print("Python :", sys.version.split()[0])
print("NumPy  :", np.__version__)
print("pandas :", pd.__version__)
print("MNE    :", mne.__version__)

## 1. Chemins, listes et dictionnaires

Les données du projet seront rangées selon la convention **BIDS** : un dossier par sujet (`sub-01`, `sub-02`, ...), les résultats dans `derivatives/`.

### Bloc Type 1 — Une arborescence de sujets fictifs

In [ ]:
# ---- 6 sujets fictifs (dossiers vides) et un dossier de résultats
root = Path("donnees_seance4")
for i in range(1, 7):
    (root / f"sub-{i:02d}" / "eeg").mkdir(parents=True, exist_ok=True)

deriv = root / "derivatives" / "seance4"
deriv.mkdir(parents=True, exist_ok=True)

dossiers_sujets = sorted(root.glob("sub-*"))
subjects = [d.name.replace("sub-", "") for d in dossiers_sujets]
print("Sujets    :", subjects)
print("Résultats :", deriv)

<details><summary>Syntaxe</summary>

- `root / "sub-01"` assemble un chemin, sur Windows comme sur macOS.
- `f"sub-{i:02d}"` écrit le numéro sur deux chiffres : `01`, `02`, ...
- `root.glob("sub-*")` liste les dossiers dont le nom commence par `sub-`.
</details>

### Bloc Type 1 — Conditions et fenêtres temporelles

Un dictionnaire associe un nom à une valeur. Les conditions et les codes ci-dessous sont **inventés** pour l'exercice.

In [ ]:
# ---- Conditions (nom -> code) et fenêtres ERP (en secondes)
event_id = {
    "assis/mot": 11,
    "assis/pseudo-mot": 12,
    "marche/mot": 21,
    "marche/pseudo-mot": 22,
}
fenetres = {
    "P200": (0.15, 0.25),
    "N400": (0.30, 0.50),
}

for nom, code_evt in event_id.items():
    print(f"{nom:<20} -> code {code_evt}")

In [ ]:
# ---- Sélectionner des conditions par leur nom
conditions_marche = [nom for nom in event_id if nom.startswith("marche")]
print(conditions_marche)

### Bloc Type 2 — À vous de compléter

1. Construisez le chemin du dossier `eeg` d'un autre sujet.
2. Ajoutez une fenêtre `"P600"` de 0,50 à 0,80 s.
3. Créez la liste des conditions **pseudo-mot**.

In [ ]:
# ---- Exercice
subject = "03"                                  # <--- essayez un autre sujet
dossier_eeg = root / f"sub-{subject}" / "eeg"
print(dossier_eeg, "existe :", dossier_eeg.exists())

# fenetres["P600"] = ...                        # <--- à compléter
# conditions_pseudo = ...                       # <--- à compléter

In [ ]:
#@title Solution (à consulter après avoir essayé)
fenetres["P600"] = (0.50, 0.80)
conditions_pseudo = [nom for nom in event_id if nom.endswith("pseudo-mot")]
print(fenetres)
print(conditions_pseudo)

## 2. L'EEG est un tableau NumPy

Jeu public **kiloword** (Dufau et al., 2015) : ERP moyens pour la lecture de 960 mots anglais, avec les caractéristiques de chaque mot (fréquence, concrétude, ...). Ce ne sont **pas** les données du projet.

### Bloc Type 1 — Charger les données

Environ 25 Mo à télécharger la première fois. Sans connexion, la cellule crée des données simulées de même format.

In [ ]:
#@title Chargement de kiloword (exécuter ; pas besoin de lire le code)
def charger_kiloword():
    try:
        chemin = mne.datasets.kiloword.data_path() / "kword_metadata-epo.fif"
        return mne.read_epochs(chemin, preload=True), "kiloword"
    except Exception as err:
        print("Téléchargement impossible, données simulées :", type(err).__name__)
        rng = np.random.default_rng(0)
        sfreq, times = 250.0, np.arange(-0.1, 0.924, 1 / 250)
        noms = ["Fz", "Cz", "Pz", "C3", "C4", "P3", "P4", "O1", "O2", "F3", "F4"]
        n = 960
        meta = pd.DataFrame({
            "WORD": [f"mot{i}" for i in range(n)],
            "Concreteness": rng.uniform(1, 7, n),
            "WordFrequency": rng.uniform(0, 4, n),
            "NumberOfLetters": rng.integers(3, 10, n).astype(float),
        })
        n400 = np.exp(-((times - 0.4) ** 2) / (2 * 0.07 ** 2))
        ampl = (-2 + 0.8 * meta["WordFrequency"].to_numpy())[:, None] * 1e-6
        data = ampl[:, None, :] * n400 + 1.5e-6 * rng.standard_normal((n, len(noms), len(times)))
        info = mne.create_info(noms, sfreq, "eeg")
        return mne.EpochsArray(data, info, tmin=times[0], metadata=meta), "simulé"

epochs, source = charger_kiloword()
print("Source :", source)
print(epochs)

### Bloc Type 1 — De l'objet MNE au tableau NumPy

`epochs.get_data()` renvoie un tableau `(essais, canaux, temps)`, en **volts**.

In [ ]:
# ---- Forme du tableau
data = epochs.get_data()
times = epochs.times
print("Forme           :", data.shape)
print("Fréq. d'échant. :", epochs.info["sfreq"], "Hz")
print("Temps           :", times[0], "à", times[-1], "s")

In [ ]:
# ---- Un essai, un canal
idx_pz = epochs.ch_names.index("Pz")
signal_pz = data[0, idx_pz, :]                  # essai 0, canal Pz, tous les temps

plt.figure(figsize=(8, 3))
plt.plot(times, signal_pz * 1e6)                # volts -> microvolts
plt.axvline(0, color="k", lw=0.8)
plt.xlabel("Temps (s)"); plt.ylabel("µV"); plt.title(f"Essai 0, Pz : « {epochs.metadata['WORD'].iloc[0]} »")
plt.show()

### Bloc Type 1 — Garder une fenêtre temporelle

Un **masque** (`True`/`False` pour chaque échantillon) garde seulement les temps de la fenêtre.

In [ ]:
# ---- Amplitude moyenne N400 à Pz, pour chaque essai
tmin, tmax = fenetres["N400"]
masque = (times >= tmin) & (times <= tmax)

n400_pz = data[:, idx_pz, masque].mean(axis=1) * 1e6   # une valeur par essai, en µV
print("Forme :", n400_pz.shape)
print("Moyenne sur les mots : %.2f µV" % n400_pz.mean())

### Bloc Type 2 — À vous de compléter

Même calcul pour la fenêtre **P200** au canal **Cz**.

In [ ]:
# ---- Exercice
# idx_cz = ...                                   # <--- à compléter
# masque_p200 = ...                              # <--- à compléter
# p200_cz = ...                                  # <--- à compléter

In [ ]:
#@title Solution (à consulter après avoir essayé)
idx_cz = epochs.ch_names.index("Cz")
tmin, tmax = fenetres["P200"]
masque_p200 = (times >= tmin) & (times <= tmax)
p200_cz = data[:, idx_cz, masque_p200].mean(axis=1) * 1e6
print("P200 moyenne à Cz : %.2f µV" % p200_cz.mean())

## 3. Fonctions, boucles et tableau de résultats

Une **fonction** regroupe le calcul ; une **boucle** l'applique plusieurs fois ; chaque résultat devient une ligne d'un tableau pandas.

### Bloc Type 1 — Une fonction

In [ ]:
# ---- Amplitude moyenne (µV) par essai
def amplitude_moyenne(epochs, canal, tmin, tmax):
    idx = epochs.ch_names.index(canal)
    masque = (epochs.times >= tmin) & (epochs.times <= tmax)
    return epochs.get_data()[:, idx, masque].mean(axis=1) * 1e6

print(amplitude_moyenne(epochs, "Pz", 0.30, 0.50)[:5])

### Bloc Type 1 — Une boucle et un tableau

In [ ]:
# ---- Une ligne par (canal, fenêtre)
lignes = []
for canal in ["Fz", "Cz", "Pz"]:
    for nom_fenetre, (tmin, tmax) in fenetres.items():
        valeurs = amplitude_moyenne(epochs, canal, tmin, tmax)
        lignes.append({"canal": canal, "fenetre": nom_fenetre, "amplitude_uV": valeurs.mean()})

resume = pd.DataFrame(lignes)
resume

<details><summary>Lien avec les notebooks d'analyse</summary>

C'est la structure de la fonction `process_subject_features` des notebooks 2025 : une fonction par sujet, une ligne par résultat, puis un `DataFrame`.
</details>

### Bloc Type 1 — Ajouter les caractéristiques des mots

`epochs.metadata` est déjà un tableau pandas (une ligne par mot). On y ajoute l'amplitude N400 et on compare les mots peu fréquents et fréquents : les mots peu fréquents produisent en général une N400 plus négative. Dans ce jeu de données, la différence à Pz est faible ; la concrétude (exercice ci-dessous) donne un effet plus net.

In [ ]:
# ---- Tableau par mot
df = epochs.metadata.copy()
df["N400_Pz_uV"] = amplitude_moyenne(epochs, "Pz", *fenetres["N400"])
df["frequence"] = pd.qcut(df["WordFrequency"], 2, labels=["faible", "élevée"])  # coupe à la médiane

df[["WORD", "WordFrequency", "frequence", "N400_Pz_uV"]].head()

In [ ]:
# ---- Moyenne par groupe
df.groupby("frequence", observed=True)["N400_Pz_uV"].agg(["mean", "std", "count"])

In [ ]:
# ---- Enregistrer en CSV et relire
fichier_csv = deriv / "n400_par_mot.csv"
df.to_csv(fichier_csv, index=False)
relu = pd.read_csv(fichier_csv)
print(fichier_csv, "->", relu.shape, "lignes x colonnes")

### Bloc Type 1 — ERP moyen par groupe de mots

On sélectionne les essais par une requête sur les métadonnées, puis on moyenne sur les essais (axe 0).

In [ ]:
# ---- Pz : mots peu fréquents vs fréquents
mediane = df["WordFrequency"].median()
plt.figure(figsize=(8, 3))
for etiquette, requete in [("faible", f"WordFrequency < {mediane}"), ("élevée", f"WordFrequency >= {mediane}")]:
    erp = epochs[requete].get_data()[:, idx_pz, :].mean(axis=0) * 1e6
    plt.plot(times, erp, label=f"fréquence {etiquette}")
plt.axvspan(*fenetres["N400"], color="0.9")
plt.axvline(0, color="k", lw=0.8)
plt.gca().invert_yaxis()                        # convention ERP : négatif vers le haut
plt.xlabel("Temps (s)"); plt.ylabel("µV"); plt.title("Pz"); plt.legend()
plt.show()

### Bloc Type 2 — À vous de compléter

Même comparaison avec la **concrétude** (`Concreteness`) : colonne de groupes, `groupby`, figure.

In [ ]:
# ---- Exercice
# df["concretude"] = ...                          # <--- à compléter

In [ ]:
#@title Solution (à consulter après avoir essayé)
df["concretude"] = pd.qcut(df["Concreteness"], 2, labels=["abstrait", "concret"])
print(df.groupby("concretude", observed=True)["N400_Pz_uV"].agg(["mean", "std", "count"]))
med = df["Concreteness"].median()
plt.figure(figsize=(8, 3))
for etiquette, requete in [("abstrait", f"Concreteness < {med}"), ("concret", f"Concreteness >= {med}")]:
    plt.plot(times, epochs[requete].get_data()[:, idx_pz, :].mean(axis=0) * 1e6, label=etiquette)
plt.gca().invert_yaxis(); plt.legend(); plt.xlabel("Temps (s)"); plt.ylabel("µV"); plt.show()

### Bloc Type 3 — Pour aller plus loin

Tracez l'amplitude N400 en fonction de `WordFrequency` (nuage de points) et calculez la corrélation : `df["WordFrequency"].corr(df["N400_Pz_uV"])`.

## 4. Aligner l'EEG et la plateforme de force

Pendant la marche, l'EEG et la plateforme de force enregistrent en même temps, mais avec **des fréquences d'échantillonnage différentes** et **des débuts différents**. Pour étudier l'EEG autour de chaque pas :

1. détecter les contacts du talon dans le signal de force ;
2. convertir ces instants dans le temps de l'EEG ;
3. découper l'EEG autour de ces instants.

> Les données ci-dessous sont **simulées** (EEG à 300 Hz, force à 1000 Hz, force qui démarre 2 s après l'EEG). Ce sont des valeurs d'exemple.

### Bloc Type 1 — Les deux signaux

In [ ]:
#@title Simulation des données de marche (exécuter ; pas besoin de lire le code)
rng = np.random.default_rng(1)
duree = 60.0                                   # secondes de marche

# Instants des contacts du talon gauche (une foulée toutes les 1,1 s environ)
pas = np.cumsum(rng.normal(1.10, 0.03, 60))
pas = pas[pas < duree - 1]

# Force verticale sous le pied gauche : 1000 Hz, démarre 2,0 s après l'EEG (horloge commune)
fs_force, t0_force = 1000, 2.0
t_force = t0_force + np.arange(0, duree, 1 / fs_force)
force = np.zeros_like(t_force)
for t_pas in pas + t0_force:                   # une bosse de force par appui (0,65 s)
    phase = (t_force - t_pas) / 0.65
    appui = (phase >= 0) & (phase <= 1)
    force[appui] += 700 * np.sin(np.pi * phase[appui])
force += rng.normal(0, 5, force.size)

# EEG : 300 Hz, 4 canaux, bruit + alpha + artefact lié à chaque pas
fs_eeg, t0_eeg = 300, 0.0
t_eeg = t0_eeg + np.arange(0, duree + t0_force, 1 / fs_eeg)
noms_eeg = ["Fz", "Cz", "Pz", "Oz"]
eeg = rng.normal(0, 5e-6, (len(noms_eeg), t_eeg.size))
phase_alpha = np.cumsum(rng.normal(0, 0.05, t_eeg.size))       # phase qui dérive lentement
eeg += 4e-6 * np.sin(2 * np.pi * 10 * t_eeg + phase_alpha)      # rythme alpha
for t_pas in pas + t0_force:
    apres = t_eeg - t_pas
    fen = (apres >= 0) & (apres < 0.3)
    eeg[:, fen] += 15e-6 * np.exp(-apres[fen] / 0.05) * np.array([[1.0], [0.6], [0.4], [0.8]])

print(f"EEG   : {eeg.shape}, {fs_eeg} Hz, début à {t0_eeg} s")
print(f"Force : {force.shape}, {fs_force} Hz, début à {t0_force} s")

In [ ]:
# ---- 5 secondes des deux signaux, sur un axe de temps commun
fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axes[0].plot(t_force, force); axes[0].set_ylabel("Force (N)")
axes[1].plot(t_eeg, eeg[0] * 1e6); axes[1].set_ylabel("Fz (µV)")
axes[1].set_xlim(10, 15); axes[1].set_xlabel("Temps (s)")
plt.show()

### Bloc Type 1 — Détecter les contacts du talon

Un contact = le moment où la force **dépasse un seuil en montant**.

In [ ]:
# ---- Franchissements du seuil
seuil = 50                                             # newtons
au_dessus = force > seuil                              # True pendant l'appui
debuts = np.where(np.diff(au_dessus.astype(int)) == 1)[0] + 1
t_contacts = t_force[debuts]                           # instants en secondes
print(len(t_contacts), "franchissements bruts")

Le bruit fait franchir le seuil plusieurs fois au même appui : on garde **une détection par foulée**.

In [ ]:
# ---- Une détection par foulée
def une_par_foulee(instants, ecart_min=0.8):
    gardes = []
    for t in instants:
        if len(gardes) == 0 or t - gardes[-1] > ecart_min:
            gardes.append(t)
    return np.array(gardes)

t_contacts = une_par_foulee(t_contacts)
print(len(t_contacts), "contacts détectés ; premiers :", np.round(t_contacts[:4], 3))

### Bloc Type 1 — Découper l'EEG autour des pas avec MNE

Tableau → `Raw` → une **annotation** par contact → événements → `Epochs` → moyenne. Ce sont les mêmes fonctions que dans les notebooks d'analyse.

In [ ]:
# ---- Raw et annotations
info = mne.create_info(noms_eeg, fs_eeg, "eeg")
raw = mne.io.RawArray(eeg, info)

onsets = t_contacts - t0_eeg                           # temps relatif au début de l'EEG
raw.set_annotations(mne.Annotations(onset=onsets, duration=0, description="heel_strike"))
print(raw)

In [ ]:
# ---- Epochs et moyenne
events, ids = mne.events_from_annotations(raw)
epochs_pas = mne.Epochs(raw, events, ids, tmin=-0.2, tmax=0.5, baseline=(-0.2, 0), preload=True)
print(epochs_pas)

moyenne_pas = epochs_pas.average()
moyenne_pas.plot(titles=dict(eeg="EEG moyen autour du contact du talon"));

La déflexion au moment du contact est un **artefact mécanique**, pas une réponse cérébrale. Pendant la marche, il s'ajoute aux réponses de la tâche lexicale ; le prétraitement sert en partie à le réduire.

### Bloc Type 2 — À vous de compléter

1. Seuil à **400 N** : le nombre de contacts change-t-il ? Et la latence de l'artefact ?
2. Remplacez `t0_eeg` par `2.0` (on oublie la conversion). Que devient la moyenne ?

In [ ]:
# ---- Exercice
seuil_test = 50                                        # <--- essayez 400
decalage_test = t0_eeg                                 # <--- essayez 2.0

debuts_test = np.where(np.diff((force > seuil_test).astype(int)) == 1)[0] + 1
onsets_test = une_par_foulee(t_force[debuts_test]) - decalage_test
onsets_test = onsets_test[(onsets_test > 0.2) & (onsets_test < raw.times[-1] - 0.5)]
raw_test = raw.copy().set_annotations(mne.Annotations(onsets_test, 0, "heel_strike"))
ev, ev_id = mne.events_from_annotations(raw_test)
mne.Epochs(raw_test, ev, ev_id, tmin=-0.2, tmax=0.5, baseline=(-0.2, 0), preload=True).average().plot();
print(len(onsets_test), "contacts utilisés")

### Bloc Type 3 — Pour aller plus loin

Rééchantillonnez la force à 300 Hz avec `np.interp(t_eeg, t_force, force)` et tracez-la sur le même axe que l'EEG.

## 5. Récapitulatif

| Brique | Usage avec les données du projet |
|---|---|
| `Path`, `sub-XX`, `derivatives/` | retrouver les fichiers, ranger les résultats |
| `event_id`, `fenetres` | conditions et fenêtres ERP |
| tableau `(essais, canaux, temps)`, masque | amplitudes et latences |
| fonction + boucle + `DataFrame` + CSV | analyse de tous les sujets |
| `Raw`, annotations, `Epochs` | découpage autour des stimuli et des pas |

Suite : `seance4_intro_mne.ipynb` (introduction à MNE).